# IMPORT LIBRAIRIES

In [1]:
import os, time, math, random
from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm
from PIL import Image
import torchvision.transforms.functional as TF

from torch.cuda.amp import autocast, GradScaler

In [2]:
class RandomPatchSigmaMapDataset(Dataset):
    """
    Retourne (inp, clean)
      inp:  (4, patch, patch) = noisy RGB + sigma_map
      clean:(3, patch, patch) = clean
    clean_dir doit contenir des images RGB.
    """
    def __init__(self, clean_dir: str, patch: int = 128, sigma_min: float = 0.0, sigma_max: float = 50.0):
        super().__init__()
        self.paths = []
        for root, _, files in os.walk(clean_dir):
            for f in files:
                if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")):
                    self.paths.append(os.path.join(root, f))
        if len(self.paths) == 0:
            raise RuntimeError(f"No images found in {clean_dir}")

        self.patch = patch
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max

    def __len__(self):
        # dataset infini "virtuel"
        return 10**9

    def _random_crop(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        p = self.patch
        if w < p or h < p:
            # upscale simple si image trop petite
            scale = max(p / w, p / h)
            nw, nh = int(round(w * scale)), int(round(h * scale))
            img = img.resize((nw, nh), Image.BICUBIC)
            w, h = img.size

        x0 = random.randint(0, w - p)
        y0 = random.randint(0, h - p)
        return img.crop((x0, y0, x0 + p, y0 + p))

    def _augment(self, img: Image.Image) -> Image.Image:
        if random.random() < 0.5:
            img = TF.hflip(img)
        if random.random() < 0.5:
            img = TF.vflip(img)
        k = random.randint(0, 3)
        if k:
            img = img.rotate(90 * k, expand=False)
        return img

    def __getitem__(self, idx):
        path = random.choice(self.paths)
        img = Image.open(path).convert("RGB")
        img = self._random_crop(img)
        img = self._augment(img)

        clean = TF.to_tensor(img)  # (3,H,W) in [0,1]

        sigma = random.uniform(self.sigma_min, self.sigma_max)  # "pixel space" 0..50
        noise = torch.randn_like(clean) * (sigma / 255.0)
        noisy = (clean + noise).clamp(0.0, 1.0)

        sigma_map = torch.full((1, clean.shape[1], clean.shape[2]), sigma / 255.0)
        inp = torch.cat([noisy, sigma_map], dim=0)  # (4,H,W)

        return inp, clean


In [3]:
# -------------------------
# DRUNet-like blocks
# -------------------------
class ResBlock(nn.Module):
    def __init__(self, nc: int):
        super().__init__()
        self.c1 = nn.Conv2d(nc, nc, 3, 1, 1)
        self.c2 = nn.Conv2d(nc, nc, 3, 1, 1)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        y = self.act(self.c1(x))
        y = self.c2(y)
        return x + y


class Down(nn.Module):
    """Downsample x2 via conv stride=2"""
    def __init__(self, in_nc: int, out_nc: int):
        super().__init__()
        self.conv = nn.Conv2d(in_nc, out_nc, 3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class Up(nn.Module):
    """Upsample x2 via PixelShuffle"""
    def __init__(self, in_nc: int, out_nc: int):
        super().__init__()
        self.conv = nn.Conv2d(in_nc, out_nc * 4, 3, 1, 1)
        self.ps = nn.PixelShuffle(2)

    def forward(self, x):
        return self.ps(self.conv(x))


def _pad_to_multiple(x: torch.Tensor, mult: int = 8) -> Tuple[torch.Tensor, Tuple[int,int,int,int]]:
    """
    Reflect pad pour que H,W soient multiples de mult.
    Retourne (x_pad, pads=(pl,pr,pt,pb)) pour pouvoir unpad.
    """
    _, _, h, w = x.shape
    pad_h = (mult - h % mult) % mult
    pad_w = (mult - w % mult) % mult
    pt = pad_h // 2
    pb = pad_h - pt
    pl = pad_w // 2
    pr = pad_w - pl
    if pad_h or pad_w:
        x = F.pad(x, (pl, pr, pt, pb), mode="reflect")
    return x, (pl, pr, pt, pb)


def _unpad(x: torch.Tensor, pads: Tuple[int,int,int,int]) -> torch.Tensor:
    pl, pr, pt, pb = pads
    if (pl, pr, pt, pb) == (0,0,0,0):
        return x
    return x[:, :, pt:x.shape[2]-pb, pl:x.shape[3]-pr]



In [4]:
# -------------------------
# Model: DRUNetSigmaMap (drop-in)
# -------------------------
class DRUNetSigmaMap(nn.Module):
    """
    Input:  (B,4,H,W) = noisy RGB + sigma map
    Output: (B,3,H,W) clean
    Residual learning: predict noise, then clean = noisy - pred_noise
    """
    def __init__(self, base: int = 64, n_res: int = 2):
        super().__init__()

        # Encoder channels
        c1, c2, c3, c4 = base, base*2, base*4, base*8

        self.head = nn.Conv2d(4, c1, 3, 1, 1)

        # level 1
        self.e1 = nn.Sequential(*[ResBlock(c1) for _ in range(n_res)])
        self.d1 = Down(c1, c2)

        # level 2
        self.e2 = nn.Sequential(*[ResBlock(c2) for _ in range(n_res)])
        self.d2 = Down(c2, c3)

        # level 3
        self.e3 = nn.Sequential(*[ResBlock(c3) for _ in range(n_res)])
        self.d3 = Down(c3, c4)

        # bottleneck
        self.mid = nn.Sequential(*[ResBlock(c4) for _ in range(max(2, n_res))])

        # Decoder
        self.u3 = Up(c4, c3)
        self.c3 = nn.Conv2d(c3 + c3, c3, 3, 1, 1)
        self.p3 = nn.Sequential(*[ResBlock(c3) for _ in range(n_res)])

        self.u2 = Up(c3, c2)
        self.c2 = nn.Conv2d(c2 + c2, c2, 3, 1, 1)
        self.p2 = nn.Sequential(*[ResBlock(c2) for _ in range(n_res)])

        self.u1 = Up(c2, c1)
        self.c1 = nn.Conv2d(c1 + c1, c1, 3, 1, 1)
        self.p1 = nn.Sequential(*[ResBlock(c1) for _ in range(n_res)])

        # Tail predicts noise (3 channels)
        self.tail = nn.Conv2d(c1, 3, 3, 1, 1)

    def forward(self, inp):
        # Pad pour gérer n'importe quelle taille
        x, pads = _pad_to_multiple(inp, mult=8)

        noisy = x[:, :3, :, :]

        x = self.head(x)

        x1 = self.e1(x)
        x  = self.d1(x1)

        x2 = self.e2(x)
        x  = self.d2(x2)

        x3 = self.e3(x)
        x  = self.d3(x3)

        x = self.mid(x)

        x = self.u3(x)
        x = self.c3(torch.cat([x, x3], dim=1))
        x = self.p3(x)

        x = self.u2(x)
        x = self.c2(torch.cat([x, x2], dim=1))
        x = self.p2(x)

        x = self.u1(x)
        x = self.c1(torch.cat([x, x1], dim=1))
        x = self.p1(x)

        pred_noise = self.tail(x)
        clean = (noisy - pred_noise).clamp(0.0, 1.0)

        # Unpad pour revenir à la taille originale
        clean = _unpad(clean, pads)
        return clean



In [ ]:
#Train
def train_drunet(
    clean_dir=r"./BSDS300/images/train",
    out_dir="weights_drunet_sigmap",
    patch=128,                 # recommandé: multiple de 8
    sigma_min=0.0,
    sigma_max=50.0,
    batch_size=8,
    steps_per_epoch=1000,
    max_epochs=10,
    lr0=1e-3,
    lr1=1e-4,
    plateau_epochs=5,
    log_every=50,
    base=64,
    n_res=2
):
    os.makedirs(out_dir, exist_ok=True)

    if patch % 8 != 0:
        print(f"[WARN] patch={patch} n'est pas multiple de 8 -> risque mismatch. Mets 128/96/64 etc.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    ds = RandomPatchSigmaMapDataset(clean_dir, patch=patch, sigma_min=sigma_min, sigma_max=sigma_max)
    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=(device == "cuda"),
        drop_last=True
    )

    model = DRUNetSigmaMap(base=base, n_res=n_res).to(device).train()
    opt = Adam(model.parameters(), lr=lr0)
    loss_fn = nn.MSELoss()
    scaler = GradScaler(enabled=(device == "cuda"))

    best = float("inf")
    stagnant = 0
    using_lr1 = False
    global_step = 0

    for epoch in range(1, max_epochs + 1):
        running = 0.0
        start_t = time.time()

        pbar = tqdm(total=steps_per_epoch, desc=f"Epoch {epoch}/{max_epochs}", leave=True)
        for step, (inp, clean) in enumerate(dl):
            if step >= steps_per_epoch:
                break

            inp = inp.to(device, non_blocking=True)
            clean = clean.to(device, non_blocking=True)

            noisy = inp[:, :3, :, :]
            target_noise = noisy - clean

            with autocast(enabled=(device == "cuda")):
                pred_clean = model(inp)
                pred_noise = noisy - pred_clean
                loss = loss_fn(pred_noise, target_noise)

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            running += loss.item()
            global_step += 1

            if (step + 1) % log_every == 0:
                avg_so_far = running / (step + 1)
                elapsed = time.time() - start_t
                it_s = (step + 1) / max(elapsed, 1e-9)
                pbar.set_postfix({
                    "loss": f"{avg_so_far:.5f}",
                    "lr": f"{opt.param_groups[0]['lr']:.1e}",
                    "it/s": f"{it_s:.2f}"
                })

            pbar.update(1)

        pbar.close()

        avg = running / max(1, steps_per_epoch)
        ckpt = os.path.join(out_dir, f"drunet_sigmap_epoch{epoch:02d}.pth")
        torch.save({"model": model.state_dict(), "epoch": epoch}, ckpt)
        print(f"Epoch {epoch:02d} done | avg_loss={avg:.6f} | lr={opt.param_groups[0]['lr']:.1e}")

        # LR schedule 
        if avg < best - 1e-7:
            best = avg
            stagnant = 0
        else:
            stagnant += 1

        if (not using_lr1) and stagnant >= plateau_epochs:
            for g in opt.param_groups:
                g["lr"] = lr1
            using_lr1 = True
            stagnant = 0
            print(f"Switch LR to {lr1}")

        if using_lr1 and stagnant >= plateau_epochs:
            print("Early stop: loss plateaued.")
            break

    final_path = os.path.join(out_dir, "drunet_sigmap_final.pth")
    torch.save({"model": model.state_dict()}, final_path)
    print(f"Saved: {final_path}")

train_drunet()

device: cuda


C:\Users\barra\AppData\Local\Temp\ipykernel_26772\3319366250.py:41: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device == "cuda"))
Epoch 1/10:   0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
def psnr_torch(x, y, eps=1e-8):
    mse = torch.mean((x - y) ** 2).item()
    return 10.0 * math.log10(1.0 / (mse + eps))


@torch.no_grad()
def denoise_image_pil(model, img_pil, sigma, device):
    y = TF.to_tensor(img_pil)  # (3,H,W) in [0,1]
    sigma_map = torch.full((1, y.shape[1], y.shape[2]), sigma / 255.0)
    inp = torch.cat([y, sigma_map], dim=0).unsqueeze(0).to(device)  # (1,4,H,W)
    out = model(inp).squeeze(0).cpu()
    return TF.to_pil_image(out)


@torch.no_grad()
def test_mode_A_clean_to_noisy(
    clean_path,
    ckpt_path,
    out_dir="test_outputs",
    sigma=25.0,
    seed=0
):
    os.makedirs(out_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = DRUNetSigmaMap().to(device).eval()
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["model"], strict=True)

    clean_pil = Image.open(clean_path).convert("RGB")
    clean = TF.to_tensor(clean_pil).unsqueeze(0)  # (1,3,H,W)

    g = torch.Generator().manual_seed(seed)
    noise = torch.randn(clean.shape, generator=g) * (sigma / 255.0)
    noisy = (clean + noise).clamp(0.0, 1.0)

    sigma_map = torch.full((1, 1, clean.shape[2], clean.shape[3]), sigma / 255.0)
    inp = torch.cat([noisy, sigma_map], dim=1).to(device)  # (1,4,H,W)
    den = model(inp).cpu()

    psnr_noisy = psnr_torch(noisy, clean)
    psnr_den = psnr_torch(den, clean)

    TF.to_pil_image(clean.squeeze(0)).save(os.path.join(out_dir, "clean.png"))
    TF.to_pil_image(noisy.squeeze(0)).save(os.path.join(out_dir, f"noisy_sigma{int(sigma)}.png"))
    TF.to_pil_image(den.squeeze(0)).save(os.path.join(out_dir, f"denoised_sigma{int(sigma)}.png"))

    print("Saved to:", out_dir)
    print(f"PSNR noisy   : {psnr_noisy:.2f} dB")
    print(f"PSNR denoised: {psnr_den:.2f} dB")


    # Test
    test_mode_A_clean_to_noisy(
        clean_path=r"./BSDS300/images/test/42049.jpg",
        ckpt_path=r"./weights_drunet_sigmap/drunet_sigmap_final.pth",
        out_dir="test_outputs_drunet",
        sigma=40
    )